In [6]:
%pip install stable_baselines3

  Using cached stable_baselines3-2.7.1-py3-none-any.whl.metadata (4.8 kB)
Using cached stable_baselines3-2.7.1-py3-none-any.whl (188 kB)
Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import numpy as np
import sys
import pandas as pd
from openpyxl import load_workbook, Workbook

notebook_dir = os.path.abspath('')

root_dir = os.path.abspath(os.path.join(notebook_dir, '..', '..'))

if root_dir not in sys.path:
    sys.path.append(root_dir)

from stable_baselines3 import SAC
from aml_project.environments.lettuce_greenhouse import LettuceGreenhouse
from aml_project.common.plot_gh_variables import create_trajectory_figure, plot_trajectory

# Define the exact coefficients from Morcego et al. (2023)
# reward_coefs = [c_r1 (growth), c_r_co2_2 (CO2 in-bounds)]
custom_reward_coefs = np.array([16.0, 0.0005])

# penalty_coefs = [c_r_u1 (CO2), c_r_u2 (Vent), c_r_u3 (Heat), c_r_co2_1 (CO2 bounds), c_r_T1 (Temp low), c_r_T2 (Temp high)]
custom_penalty_coefs = np.array([4.5360e-4, 0.0075, 8.5725e-4, 0.1, 0.001, 0.0005])

weather_file = 'environments/weather/outdoorWeatherWurGlas2014.mat'

env = LettuceGreenhouse(
    weather_direction=weather_file, 
    reward_coefs=custom_reward_coefs, 
    penalty_coefs=custom_penalty_coefs
)



ValueError: Key backend: 'module://matplotlib_inline.backend_inline' is not a valid value for backend; supported values are ['gtk3agg', 'gtk3cairo', 'gtk4agg', 'gtk4cairo', 'macosx', 'nbagg', 'notebook', 'qtagg', 'qtcairo', 'qt5agg', 'qt5cairo', 'tkagg', 'tkcairo', 'webagg', 'wx', 'wxagg', 'wxcairo', 'agg', 'cairo', 'pdf', 'pgf', 'ps', 'svg', 'template']

In [ ]:

# Initialize the Soft Actor-Critic model
model = SAC("MlpPolicy", env, verbose=1, learning_rate=1e-3, batch_size=64)
model.learn(total_timesteps=100)
model.save("sac_lettuce_greenhouse")

In [ ]:
# We must reset the environment before testing
obs, info = env.reset()

done = False
truncated = False


while not (done or truncated):
    # Predict the best action (deterministic=True means no random exploration)
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = env.step(action)

In [ ]:
print("Plotting results...")

y = env.y.T 
d = env.d.T 

u = env.u[:, :-1].T 

nvars = 11
n_per_day = int(24 * 3600 / env.h) # control frequency per day
label = 'Trained SAC Policy'


trajectory = np.concatenate((y, d, u), axis=1)

# Generate the plot
fig, axes = create_trajectory_figure(nvars, env.L, env.h, env.c, None, None)
fig, axes = plot_trajectory(fig, axes, trajectory, env.L, env.h, env.c, 0, env.n_days, n_per_day, label)

# Show the plot
import matplotlib.pyplot as plt
plt.show()